In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

try:
    response = groq_client.chat.completions.create(
        # Llama-3.1-70b ki jagah naya Llama-3.3 ya 8b use karen
        model="llama-3.3-70b-versatile",  # Ya "llama-3.1-8b-instant" test karne ke liye
        messages=[
            {"role": "user", "content": "Hi! Mujhe batao ke RAG pipeline kya hoti hai short mein."}
        ]
    )
    print("Groq ka Jawab:")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"Error: {e}")

Groq ka Jawab:
RAG (Retrieve, Augment, Generate) pipeline ek AI model hai jo question-answering tasks ko perform karne ke liye design kiya gaya hai. Yah pipeline do pramukh components se banaya gaya hai:

1. **Retrieve**: Iss phase mein, model ek prashn (question) ke liye related text ya documents ko dhundhta hai. Yah text ya documents ek badi database mein se retrieve kiye jate hain.
2. **Augment aur Generate**: Jab related text ya documents mil jaate hain, to model unhein read karta hai aur prashn ka uttar (answer) generate karta hai. Iss phase mein, model uttar ko refine karne ke liye additional information bhi jod sakta hai.

RAG pipeline ka mukhya uddeshya hai prashn-uttar tasks ko adhik kushal aur sahi tarike se perform karna. Yah pipeline question-answering, text summarization, aur chatbot jaise applications mein upyog kiya ja sakta hai.


In [12]:
def llm(prompt):
    # Jo client humne upar banaya tha (groq_client) use call karenge
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile", # Active Groq model
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    # Jawab nikalne ka sahi tareeqa
    return response.choices[0].message.content

# Ab isay run karen
print(llm('What is RAG pipeline? Explain in short.'))

The RAG (Retrieve, Augment, Generate) pipeline is a natural language processing (NLP) and machine learning framework. 

1. **Retrieve**: It fetches relevant information from a database or knowledge base.
2. **Augment**: It combines the retrieved information with the input prompt or question.
3. **Generate**: It uses the augmented information to generate a response or answer.

The RAG pipeline is often used in question-answering, text summarization, and other NLP tasks.


In [13]:
llm('Hi')

"It's nice to meet you. Is there something I can help you with, or would you like to chat?"

In [15]:
question = 'I Just Discovered the Course, Can i Join Now?'
answer = llm(question)
print(answer)

I'm excited you're interested in joining the course. However, I need a bit more information from you. 

Could you please provide more details about the course you're referring to, such as:
1. What is the course about?
2. Is it an online or offline course?
3. Are there any specific enrollment deadlines or requirements?

Once I have more information, I'll be happy to help you determine if you can join the course now.


In [20]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [21]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [22]:
question = 'I Just Discovered the Course, Can i Join Now?'
answer = llm(prompt)
print(answer)

Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Additionally, you don't need a confirmation email to start, you can just begin learning and submitting homework.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [23]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [25]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1349

In [26]:
documents [10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [27]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [28]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [29]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [30]:
search_results = search(question)


In [31]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [3]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [12]:
USER_PROMPT_TEMPLATE = """
question:
{question}

Context:
{context}
"""

In [14]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [15]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [16]:
prompt = build_prompt(question, search_results)

print(prompt)

question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project...
